# Preprocessing

Replicates the preprocessing pipeline from Hilders et al `data_processing.ipynb`.

1. Outlier removal
2. Forward-fill (max 24h)
3. Drop features >60% missing
4. Drop patients >30% missing
5. Median imputation
6. Label derivation

Most of the code is taken from Hilders et al. data_processing.ipynb. Claude AI and copilot was used in debugging errors.

Input: `data/features/features_raw.csv`  
Output: `data/features/features_processed.csv`

In [ ]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

input_path  = os.path.join('..', 'data', 'features', 'features_raw.csv')
output_path = os.path.join('..', 'data', 'features', 'features_processed.csv')
cohort_path = os.path.join('..', 'data', 'cohort', 'sepsis3_cohort.csv')

In [ ]:
df = pd.read_csv(input_path)
print(f'{df.admissionid.nunique()} admissions, {len(df):,} rows, {len(df.columns)} columns')

# Save rrt separately before any dropping - needed for rrt_next_timestep label
rrt_saved = df[['admissionid', 'window', 'rrt']].copy() if 'rrt' in df.columns else None
df.head()

## Outlier removal



In [ ]:
# Source: Hilders et al. data_processing.ipynb features_filter dictionary
bounds = {
    'albumin': (0, 50), 'alt': (0, 6000), 'ast': (0, 20000),
    'bicarbonate': (0, 50), 'bilirubin': (0, 500), 'bun': (0, 100),
    'calcium': (0, 5), 'potassium': (1, 9), 'chloride': (50, 125),
    'crp': (0, 500), 'creatinine': (0, 1200), 'glucose': (0, 40),
    'heart_rate': (30, 180), 'hematocrit': (0.100, 0.600),
    'hemoglobin': (0, 15), 'inr': (0, 15), 'lactate': (0, 30),
    'spo2': (74, 100), 'pao2': (20, 400), 'paco2': (15, 150),
    'ph': (7.0, 8.5), 'platelet': (0, 1000), 'resp_rate': (5, 50),
    'sodium': (115, 175), 'sysbp': (40, 300), 'meanbp': (20, 160),
    'diasbp': (10, 130), 'cvp': (-10, 30), 'tempc': (33, 42),
    'wbc': (0, 100), 'gcs_eye': (1, 4), 'gcs_verbal': (1, 5),
    'gcs_motor': (1, 6), 'peep': (0, 30),
}

removed = {}
for col, (lo, hi) in bounds.items():
    if col not in df.columns:
        continue
    n = df[col].notna().sum()
    df.loc[df[col] < lo, col] = np.nan
    df.loc[df[col] > hi, col] = np.nan
    diff = n - df[col].notna().sum()
    if diff > 0:
        removed[col] = diff

print('Outliers removed:')
for c, n in sorted(removed.items(), key=lambda x: -x[1]):
    print(f'  {c}: {n:,}')

In [ ]:
# No vasopressor/fluid = 0, not missing
df['vaso_rate'].fillna(0.0, inplace=True)
df['fluid_window'].fillna(0.0, inplace=True)

# Values above 3.0 were likely entered as micrograms instead of milligrams
df['vaso_rate'] = np.where(df['vaso_rate'] > 3.0, df['vaso_rate'] / 1000, df['vaso_rate'])

## Forward-fill (max 6 windows = 24h)

In [ ]:
df = df.sort_values(['admissionid', 'window']).reset_index(drop=True)

static_cols = ['admissionid', 'window', 'start_time', 'gender', 'agegroup', 'weightgroup']
fill_cols = [c for c in df.columns if c not in static_cols]

before = df[fill_cols].isna().sum().sum()
df[fill_cols] = df.groupby('admissionid')[fill_cols].ffill(limit=6)
after = df[fill_cols].isna().sum().sum()

print(f'Missing before: {before:,}  after: {after:,}  filled: {before-after:,}')

## Feature and patient exclusion

In [ ]:
# Source: Hilders et al. data_processing.ipynb - 60% feature threshold, 30% patient threshold
clinical_cols = list(bounds.keys()) + ['vaso_rate', 'fluid_window', 'rrt']
clinical_cols = [c for c in clinical_cols if c in df.columns]

miss_rate = df[clinical_cols].isna().sum() / len(df)
print('Missingness after forward-fill:')
print(miss_rate.sort_values(ascending=False).round(3).to_string())

dropped_feats = miss_rate[miss_rate > 0.60].index.tolist()
df.drop(columns=dropped_feats, inplace=True)
clinical_cols = [c for c in clinical_cols if c not in dropped_feats]
print(f'\nDropped features (>60% missing): {dropped_feats}')
print(f'Remaining: {len(clinical_cols)}')

In [ ]:
n_before = df.admissionid.nunique()

miss_per_pt = df.groupby('admissionid')[clinical_cols].apply(lambda x: x.isna().sum().sum())
total_per_pt = df.groupby('admissionid')[clinical_cols].apply(lambda x: x.shape[0] * x.shape[1])
miss_pct = miss_per_pt / total_per_pt

print('Patient missingness distribution:')
print(miss_pct.describe().round(3))

to_drop = miss_pct[miss_pct > 0.30].index
df = df[~df['admissionid'].isin(to_drop)]
print(f'\nPatients: {n_before} → {df.admissionid.nunique()} (dropped {len(to_drop)})')

## Median imputation

Computed on the full dataset before splitting. Minor leakage acknowledged - same approach as Hilders et al.

In [ ]:
# Source: Hilders et al. data_processing.ipynb
from sklearn.impute import SimpleImputer
imp = SimpleImputer(missing_values=np.nan, strategy='median')
imp.fit(df[clinical_cols])
df[clinical_cols] = imp.transform(df[clinical_cols])

print('Median imputation applied using sklearn SimpleImputer')
print(f'Remaining missing: {df[clinical_cols].isna().sum().sum()}')

## Label derivation



In [ ]:
# Source: Hilders et al. data_processing.ipynb label derivation logic
df['start_time'] = pd.to_datetime(df['start_time'])
df = df.reset_index(drop=True)
window_end = df['start_time'] + pd.Timedelta(hours=4)

cohort = pd.read_csv(cohort_path)

# 90-day mortality
df = df.merge(cohort[['admissionid','dateofdeath']].drop_duplicates(), on='admissionid', how='left')
df['dateofdeath'] = pd.to_datetime(df['dateofdeath'], unit='ms', errors='coerce')
df['90day_mortality'] = np.where((df['dateofdeath'] - window_end) < pd.Timedelta(days=90), 1, 0)
df.drop(columns=['dateofdeath'], inplace=True)

# Remaining LOS
df = df.merge(cohort[['admissionid','dischargedat']].drop_duplicates(), on='admissionid', how='left')
df['dischargedat'] = pd.to_datetime(df['dischargedat'], unit='ms', errors='coerce')
df['LOS_remaining_days'] = (df['dischargedat'] - window_end).dt.total_seconds() / (24 * 3600)
df.drop(columns=['dischargedat'], inplace=True)

# Next-step RRT
if rrt_saved is not None:
    rrt = rrt_saved.copy()
    rrt['rrt'] = rrt.groupby('admissionid')['rrt'].ffill(limit=6).fillna(0)
    rrt['rrt_next_timestep'] = rrt.groupby('admissionid')['rrt'].shift(-1).fillna(0).astype(int)
    df = df.merge(rrt[['admissionid','window','rrt_next_timestep']], on=['admissionid','window'], how='left')
    df['rrt_next_timestep'] = df['rrt_next_timestep'].fillna(0).astype(int)

# Next-step MV
if 'peep' in df.columns:
    df['mv_active'] = np.where((df['peep'].notna()) & (df['peep'] > 0), 1, 0)
    df['mv_next_timestep'] = df.groupby('admissionid')['mv_active'].shift(-1).fillna(0).astype(int)
    df.drop(columns=['mv_active'], inplace=True)

print('Label distributions:')
print(f'  90-day mortality:  {df["90day_mortality"].mean():.3f}')
if 'rrt_next_timestep' in df.columns:
    print(f'  Next-step RRT:     {df["rrt_next_timestep"].mean():.3f}')
if 'mv_next_timestep' in df.columns:
    print(f'  Next-step MV:      {df["mv_next_timestep"].mean():.3f}')
print(f'  Remaining LOS med: {df["LOS_remaining_days"].median():.1f} days')

In [ ]:
df.to_csv(output_path, index=False)
print(f'Saved {len(df):,} rows ({df.admissionid.nunique()} admissions)')
print(f'Columns: {list(df.columns)}')
df.head()